# Nemotron 3 Diarization — Colab T4 + Gradio UI

**Embargo:** private until 23 Sep 2026, 08:00 PT. Do not share this notebook publicly.

This notebook is how the checkpoint **actually runs**. This author has **no local NVIDIA GPU**. Runtime must be a **T4** (or larger).

1. `Runtime → Change runtime type → T4 GPU`
2. Upload `Nemotron-3-Diarization-preview.nemo` to Drive (or the file picker below)
3. Run all — the last cells attach a **Gradio** speaker-lane UI to this notebook

The `.nemo` is **not** in the git repo (Early Access).


In [ ]:
# GPU check — stop if this is not CUDA
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime → Change runtime type → T4. Demo Gradio will still launch.")
!nvidia-smi -L || true


In [ ]:
# Install (first run takes several minutes)
import sys
print(sys.version)
%pip install -q -U "nemo_toolkit[asr]" gradio soundfile librosa


In [ ]:
from pathlib import Path

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not Colab — set NEMO_PATH manually if you have a GPU box.")

NEMO_CANDIDATES = [
    Path("/content/Nemotron-3-Diarization-preview.nemo"),
    Path("/content/drive/MyDrive/Nemotron-3-Diarization-preview.nemo"),
    Path("/content/drive/MyDrive/nemotron/Nemotron-3-Diarization-preview.nemo"),
]
NEMO_PATH = next((p for p in NEMO_CANDIDATES if p.exists()), None)
if NEMO_PATH is None and IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount skipped:", e)
    NEMO_PATH = next((p for p in NEMO_CANDIDATES if p.exists()), None)

if NEMO_PATH is None and IN_COLAB:
    print("Upload the .nemo from Downloads if it is not on Drive.")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".nemo"):
            NEMO_PATH = Path("/content") / name
            break

HAS_CKPT = bool(NEMO_PATH and Path(NEMO_PATH).exists())
print("checkpoint:", NEMO_PATH)
print("HAS_CKPT", HAS_CKPT)


In [ ]:
# Load the Early Access checkpoint when CUDA + file are present
MODEL = None
if torch.cuda.is_available() and HAS_CKPT:
    from nemo.collections.asr.models import SortformerEncLabelModel
    MODEL = SortformerEncLabelModel.restore_from(str(NEMO_PATH), map_location="cuda")
    MODEL.cuda().eval()
    print("loaded", NEMO_PATH)
else:
    print("Using Demo JSON — upload the .nemo and pick a T4 to run the real model.")


In [ ]:
# Fallback session (same mix the local showcase plays)
DEMO = {"id": "launch-review-overlap", "title": "Launch review \u2014 three speakers, three overlaps", "audio": "/static/demo-mix.wav", "duration_sec": 9.022, "sample_rate": 16000, "note": "Constructed multi-speaker mix in the JSON shape the Colab notebook emits after Nemotron 3 Diarization + ASR. Overlaps are intentional.", "speakers": [{"id": "speaker_0", "name": "Priya \u00b7 Product", "color": "#76B900"}, {"id": "speaker_1", "name": "Daniel \u00b7 Engineering", "color": "#2DD4BF"}, {"id": "speaker_2", "name": "Marcus \u00b7 Legal", "color": "#F5A623"}], "segments": [{"speaker": "speaker_0", "start": 0.0, "end": 2.063, "text": "The launch window is Friday."}, {"speaker": "speaker_1", "start": 1.15, "end": 2.917, "text": "We should ship it this afternoon."}, {"speaker": "speaker_0", "start": 3.55, "end": 5.756, "text": "Legal still has to sign off."}, {"speaker": "speaker_2", "start": 4.7, "end": 8.241, "text": "Can we at least wait for the security review?"}, {"speaker": "speaker_1", "start": 7.1, "end": 8.572, "text": "The customers are waiting."}], "blended": "The launch window is Friday. We should ship it this afternoon. Legal still has to sign off. Can we at least wait for the security review? The customers are waiting.", "overlaps": [{"start": 1.15, "end": 2.063, "speakers": ["speaker_0", "speaker_1"]}, {"start": 4.7, "end": 5.756, "speakers": ["speaker_0", "speaker_2"]}, {"start": 7.1, "end": 8.241, "speakers": ["speaker_2", "speaker_1"]}]}
import json as _json
from pathlib import Path as _P
print("demo speakers", len(DEMO["speakers"]), "overlaps", len(DEMO["overlaps"]))


In [ ]:
import tempfile
import numpy as np

def diarize_file(wav_path: str):
    """Run Nemotron 3 Diarization when the GPU model is loaded."""
    if MODEL is None:
        return DEMO, "demo (no GPU checkpoint in this runtime)"
    segs = MODEL.diarize(audio=wav_path, batch_size=1)
    # NeMo returns list[list[str]] like '0.12 1.04 speaker_0'
    raw = segs[0] if segs and isinstance(segs[0], (list, tuple)) else segs
    parsed = []
    for line in raw:
        if isinstance(line, (list, tuple)):
            line = " ".join(map(str, line))
        parts = str(line).split()
        if len(parts) >= 3:
            parsed.append({
                "start": float(parts[0]),
                "end": float(parts[1]),
                "speaker": parts[2] if parts[2].startswith("speaker") else f"speaker_{parts[2]}",
                "text": "",
            })
    speakers = sorted({p["speaker"] for p in parsed})
    palette = ["#76B900", "#2DD4BF", "#F5A623", "#60A5FA", "#E879F9", "#F87171", "#A3E635", "#22D3EE"]
    out = {
        "title": Path(wav_path).name,
        "duration_sec": max((p["end"] for p in parsed), default=0),
        "speakers": [{"id": s, "name": s, "color": palette[i % len(palette)]} for i, s in enumerate(speakers)],
        "segments": parsed,
        "overlaps": [],
        "blended": "",
    }
    for i, a in enumerate(parsed):
        for b in parsed[i+1:]:
            start, end = max(a["start"], b["start"]), min(a["end"], b["end"])
            if end - start > 0.05:
                out["overlaps"].append({"start": start, "end": end, "speakers": [a["speaker"], b["speaker"]]})
    return out, "nemotron-3-diarization"


In [ ]:
# Optional: word-level ASR (Whisper) then assign words to speaker segments
def maybe_transcribe(wav_path: str, session: dict) -> dict:
    try:
        import whisper
    except Exception:
        print("Skipping ASR (whisper not installed). Diarization-only session.")
        return session
    asr = whisper.load_model("base")
    result = asr.transcribe(wav_path, word_timestamps=True)
    session = dict(session)
    session["blended"] = result.get("text", "").strip()
    words = []
    for seg in result.get("segments") or []:
        words.extend(seg.get("words") or [])
    if not words:
        return session
    for spk_seg in session["segments"]:
        bits = [w.get("word", "") for w in words if w.get("end", 0) > spk_seg["start"] and w.get("start", 0) < spk_seg["end"]]
        if bits:
            spk_seg["text"] = "".join(bits).strip()
    return session


In [ ]:
import gradio as gr

LANE_CSS = '''
.lane{display:grid;grid-template-columns:140px 1fr;gap:8px;margin:8px 0;color:#e8f0dc;font-family:system-ui,sans-serif}
.track{position:relative;height:28px;background:#0c0f0a;border:1px solid #2a3324;border-radius:8px}
.seg{position:absolute;top:4px;bottom:4px;border-radius:5px}
.ov{position:absolute;top:0;bottom:0;background:repeating-linear-gradient(-45deg,rgba(245,166,35,.2),rgba(245,166,35,.2) 4px,transparent 4px,transparent 8px)}
.turn{border-left:4px solid #76B900;padding:8px 10px;margin:6px 0;background:#12150f;color:#e8f0dc}
'''

def lanes_html(session):
    dur = max(session.get("duration_sec") or 1, 0.01)
    colors = {s["id"]: s["color"] for s in session["speakers"]}
    names = {s["id"]: s["name"] for s in session["speakers"]}
    html = [f"<style>{LANE_CSS}</style>"]
    for spk in session["speakers"]:
        html.append('<div class="lane">')
        html.append(f'<div>{names[spk["id"]]}</div><div class="track">')
        for seg in session["segments"]:
            if seg["speaker"] != spk["id"]:
                continue
            left = 100 * seg["start"] / dur
            width = 100 * (seg["end"] - seg["start"]) / dur
            html.append(f'<div class="seg" style="left:{left:.2f}%;width:{width:.2f}%;background:{colors[spk["id"]]}"></div>')
        for ov in session.get("overlaps") or []:
            if spk["id"] not in ov["speakers"]:
                continue
            left = 100 * ov["start"] / dur
            width = 100 * (ov["end"] - ov["start"]) / dur
            html.append(f'<div class="ov" style="left:{left:.2f}%;width:{width:.2f}%"></div>')
        html.append("</div></div>")
    for seg in session["segments"]:
        txt = seg.get("text") or ""
        html.append(f'<div class="turn" style="border-left-color:{colors.get(seg["speaker"], "#76B900")}"><small>{names.get(seg["speaker"], seg["speaker"])} · {seg["start"]:.2f}–{seg["end"]:.2f}s</small><br>{txt}</div>')
    n_ov = len(session.get("overlaps") or [])
    html.append(f"<p style='color:#8a947c;font-family:system-ui'>overlaps: {n_ov} · mode below</p>")
    return "".join(html)

def run(audio, do_asr):
    if audio is None:
        sess, mode = DEMO, "demo (play the built-in overlapping mix)"
        return sess["blended"], lanes_html(sess), mode
    sess, mode = diarize_file(audio)
    if do_asr and MODEL is not None:
        sess = maybe_transcribe(audio, sess)
        mode += " + whisper"
    return sess.get("blended") or "(diarization only — enable ASR or read speaker lanes)", lanes_html(sess), mode

demo = gr.Interface(
    fn=run,
    inputs=[
        gr.Audio(type="filepath", label="Multi-speaker wav (or skip to use the overlapping demo)"),
        gr.Checkbox(label="Also run Whisper ASR (needs extra RAM)", value=False),
    ],
    outputs=[
        gr.Textbox(label="Blended transcript (no speakers)"),
        gr.HTML(label="Speaker lanes · hatched = overlap"),
        gr.Textbox(label="Mode"),
    ],
    title="Nemotron 3 Diarization",
    description="T4 GPU + .nemo = live diarization. Without them this Gradio still shows the overlapping-voice demo.",
    allow_flagging="never",
)
demo.launch(share=False, debug=False)
